In [14]:
import random
import torch
from torch import nn
from torch.amp import GradScaler
from torch.utils.data import Dataset, DataLoader
from torch.utils.tensorboard import SummaryWriter
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

scaler = GradScaler(device=device)

Using device: cuda


In [15]:
# phone map

phone_map = {'aa': 0, 'ae': 1, 'ah': 2, 'ao': 3, 'aw': 4, 'ax': 5, 'ax-h': 6, 'axr': 7, 
'ay': 8, 'b': 9, 'bcl': 10, 'ch': 11, 'd': 12, 'dcl': 13, 'dh': 14, 'dx': 15, 
'eh': 16, 'el': 17, 'em': 18, 'en': 19, 'eng': 20, 'epi': 21, 'er': 22, 'ey': 23, 
'f': 24, 'g': 25, 'gcl': 26, 'h#': 27, 'hh': 28, 'hv': 29, 'ih': 30, 'ix': 31, 
'iy': 32, 'jh': 33, 'k': 34, 'kcl': 35, 'l': 36, 'm': 37, 'n': 38, 'ng': 39, 
'nx': 40, 'ow': 41, 'oy': 42, 'p': 43, 'pau': 44, 'pcl': 45, 'q': 46, 'r': 47, 
's': 48, 'sh': 49, 't': 50, 'tcl': 51, 'th': 52, 'uh': 53, 'uw': 54, 'ux': 55, 
'v': 56, 'w': 57, 'y': 58, 'z': 59, 'zh': 60}

In [16]:
# load whisper embeddings
print("Loading saved embeddings...")

whisper_train_embeddings = torch.load('data/saved_embeddings/whisper_train_embeddings.pt')
whisper_train_labels = torch.load('data/saved_embeddings/whisper_train_labels.pt')

for tensor in tqdm(whisper_train_labels, "Remapping train labels: phone to int", len(whisper_train_labels)):
    tensor.apply_(lambda x: phone_map[x])

print("Train records: " + str(len(whisper_train_embeddings)))


class AudioDataset(Dataset):
    def __init__(self, x, y):
        self.x         = x
        self.y         = y

    def __getitem__(self, index):
        return self.x[index], self.y[index]

    def __len__(self):
        return len(self.x)

whisper_train_set = AudioDataset(whisper_train_embeddings, whisper_train_labels)

whisper_train_loader = DataLoader(
    dataset=whisper_train_set,
    batch_size=64,
    shuffle=True,
    num_workers=0
)

print('Train batches: ' + str(len(whisper_train_loader)))

Loading saved embeddings...


KeyboardInterrupt: 

In [ ]:
print(whisper_train_labels[0])

h#


In [ ]:
# define probe (for whisper)

class ProbeNet(nn.Module):
    def __init__(self, embedding_dim):
        super(ProbeNet, self).__init__()

        self.layers = nn.Sequential(
            nn.Linear(embedding_dim, 200),
            nn.ReLU(),
            nn.Linear(200, 61) # 61 phones
        )
    
    def forward(self, x):
        x = self.dense(x)
        return torch.sigmoid(x)

In [ ]:
# initialize model and set up stuff
whisper_probe = ProbeNet(1280)

optimizer = torch.optim.Adam(whisper_probe.parameters())
criterion = nn.BCELoss()

In [ ]:
# define train method

def train(model, epoch, train_loader):
    model.train()

    correct = 0
    total = 0

    for data in tqdm(train_loader, desc="Training", total=len(train_loader)):
    # for data in train_loader:
        inputs = data[0].to(device)
        targets = data[1].to(device)

        outputs = model(inputs)

        loss = criterion(outputs, inputs)

        scaler.scale(loss).backwards()
        scaler.stop(optimizer)
        scaler.update()

        predicted = torch.round(outputs)
        total += targets.size()
        correct += predicted.eq(targets).sum().item()
    
    accuracy = 100.*correct / total
    writer.add_scalar('whisper probe accuracy', accuracy, epoch)

    